# **Mental Health Sentiment Analysis using BERT**

This notebook fine-tunes a BERT model to classify statements into 7 mental health categories:

 - Normal 
 - Depression
 - Suicidal 
 - Anxiety 
 - Stress
 -  Bi-Polar
 -  Personality Disorder

In [ ]:
# !pip install datasets

### **Importing Libraries**

In [ ]:


import os
import re
import string
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# NLP & ML
import nltk
import torch
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.utils import resample
from sklearn.metrics import classification_report, confusion_matrix
from nltk.corpus import stopwords
from datasets import Dataset


# Transformers
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments, EarlyStoppingCallback

# NLTK downloads
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')


stop_words = set(stopwords.words('english'))

### **Load Dataset**

In [ ]:

df = pd.read_csv('/content/data/Combined Data.csv')
df.drop(columns=['Unnamed: 0'], axis=1,  inplace = True)
df.head()

### **Preprocessing**

In [ ]:
df.isnull().sum()

In [ ]:
df.dropna(inplace=True)
df.isnull().sum()

In [ ]:
df.shape

In [ ]:
# filter N rows 
N = 10000
df = df.sample(n=N, random_state=42).reset_index(drop=True)

In [ ]:
df.shape

In [ ]:
def preprocess_text(text: str) -> str:
    """Clean and preprocess input text."""
    text = text.lower()
    text = re.sub(r'[^a-zA-Z0-9\s]', ' ', text)
    tokens = nltk.word_tokenize(text)
    tokens = [word for word in tokens if word not in stop_words and word not in string.punctuation]
    return " ".join(tokens)

In [ ]:
df['statement'].iloc[2]

In [ ]:
df['statement'] = df['statement'].apply(preprocess_text)

In [ ]:
df['statement'].iloc[2]

In [ ]:
df['status'].value_counts()

In [ ]:
# encode labels
le = LabelEncoder()
df['label_encoded'] = le.fit_transform(df['status']) 

In [ ]:
# upsample minority classes
max_size = df['label_encoded'].value_counts().max()
lst = [df]
for class_index, group in df.groupby('label_encoded'):
    lst.append(group.sample(max_size-len(group), replace=True, random_state=42))
df_balanced = pd.concat(lst).sample(frac=1, random_state=42).reset_index(drop=True)



In [ ]:
# train-test split
train_df, test_df = train_test_split(df_balanced, test_size=0.2, stratify=df_balanced['label_encoded'], random_state=42)


In [ ]:

# tokenization
MODEL_NAME = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
MAX_LEN = 200

class Dataset(torch.utils.data.Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = self.labels[idx]

        inputs = self.tokenizer(
            text,
            truncation=True,
            padding='max_length',
            max_length=self.max_len,
            return_tensors='pt'
        )

        # remove batch dim
        item = {key: val.squeeze(0) for key, val in inputs.items()}
        item['labels'] = torch.tensor(label, dtype=torch.long)
        return item



In [ ]:
# model
num_labels = len(le.classes_)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=num_labels, hidden_dropout_prob=0.2)


In [ ]:

train_dataset = Dataset(train_df['statement'].tolist(), train_df['label_encoded'].tolist(), tokenizer, MAX_LEN)
test_dataset = Dataset(test_df['statement'].tolist(), test_df['label_encoded'].tolist(), tokenizer, MAX_LEN)

In [ ]:


# trainingArguments
training_args = TrainingArguments(
    output_dir='./results',
    eval_strategy='epoch',
    save_strategy='epoch',
    optim="adamw_torch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=5,
    weight_decay=0.01,
    logging_dir='./logs',
    logging_steps=10,
    lr_scheduler_type='linear',
    warmup_steps=500,
    metric_for_best_model='eval_loss',
    load_best_model_at_end=True,
    save_total_limit=4,
    gradient_accumulation_steps=2,
    fp16=torch.cuda.is_available(), 
)



In [ ]:

# trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

# train
trainer.train()


In [ ]:
def predict_sentiment(text):
    inputs = tokenizer(
        text,
        truncation=True,
        padding='max_length',
        max_length=MAX_LEN,
        return_tensors='pt'
    )

    device = next(model.parameters()).device
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)
    logits = outputs.logits
    prediction_label = torch.argmax(logits, dim=1).item()
    return le.inverse_transform([prediction_label])[0]


In [ ]:
texts = ["I love spending time with my friends.", 
         "I’m really stressed about my exams."
         "I can’t take this pain anymore."
          "I want to end my life."         
         ]

In [ ]:
for t in texts:
    print(f"{t} → {predict_sentiment(t)}")

In [ ]:
pred_output = trainer.predict(test_dataset)
predictions = pred_output.predictions
labels = pred_output.label_ids
prediction_labels = np.argmax(predictions, axis=1)

print(classification_report(labels, prediction_labels, target_names=le.classes_))

cm = confusion_matrix(labels, prediction_labels)
plt.figure(figsize=(10, 7))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=le.classes_, yticklabels=le.classes_)
plt.title('Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()